# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [49]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import sys
import os


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [50]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 50
EPOCHS = 25
LR = 1e-4
num_classes = 50

notebook_dir = os.getcwd()

data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

## **1. Preprocessing**

In [51]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# preprocessor = TorchPreprocessor(
#     normalize=True,
#     mean = IMAGENET_MEAN,
#     std = IMAGENET_STD,
#     resize_method="pad",
#     target_size=(224, 224),
# )

# train_dataset, val_dataset = train_val_split(train_transform=preprocessor, val_transform=preprocessor)


In [52]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [53]:
heavy_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="heavy",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

light_training_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="light",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)

val_preprocessor = TorchPreprocessor(
    normalize=True,
    mean = IMAGENET_MEAN,
    std = IMAGENET_STD,
    augmentation="none",
    resize_method="pad",
    target_size=(224, 224),
    interpolation_method="BICUBIC",
)


train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, target_size=(224, 224), batch_size=BATCH_SIZE, train_preprocessor_light=light_training_preprocessor, train_preprocessor_heavy=heavy_training_preprocessor, val_preprocessor=val_preprocessor)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


## **2. Modèle**

In [54]:
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
import torch.nn as nn

def create_model(num_classes: int) -> nn.Module:
    model = timm.create_model('tf_efficientnetv2_b0.in1k', pretrained=True)
    
    in_features = model.get_classifier().in_features

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4), # 0.2 est suffisant pour un petit modèle B0
        nn.Linear(in_features, num_classes)
    )
    
    return model

# Initialisation
model = create_model(num_classes)
model.to(DEVICE)
criterion = nn.CrossEntropyLoss()

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LR,
    weight_decay=1e-4
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

## **3. F1-score**

In [55]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [56]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()


        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, num_classes)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [57]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_per_class = []
    for cls in range(num_classes):
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))
        
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
        
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)
    
    f1_macro = np.mean(f1_per_class)

    return total_loss / len(val_loader), f1_macro, f1_per_class

## **5. Entrainement**

**Vérif des labels**

In [58]:
# all_labels = [label for _, label in train_dataset]
# print("Label min:", min(all_labels))
# print("Label max:", max(all_labels))
# print("Nombre de classes uniques:", len(set(all_labels)))

# # Récupérer tous les labels uniques triés
# all_labels = sorted(set(label for _, label in train_dataset.samples))
# label_to_index = {label: i for i, label in enumerate(all_labels)}

# # Remapper les labels dans le dataset
# # for i, (path, label) in enumerate(train_dataset.samples):
# #     train_dataset.samples[i] = (path, label_to_index[label])

**Entrainement**

In [59]:
import csv

# --- CONFIGURATION INITIALE ---
FREEZE_EPOCHS = 3  # Nombre d'époques où seul le classifier travaille
best_f1 = 0.0
best_model_path = "best_model.pth"
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro', 'status']

# 1. On commence par geler le modèle (Backbone)
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters(): # Spécifique à EffNet
    param.requires_grad = True


with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

# --- BOUCLE D'ENTRAÎNEMENT ---
for epoch in range(EPOCHS):
    
    # 2. Gestion du dégel (Fine-tuning) après FREEZE_EPOCHS
    status = "Frozen"
    if epoch == FREEZE_EPOCHS:
        print("\n>>> Dégel du modèle complet pour le Fine-tuning !")
        for param in model.parameters():
            param.requires_grad = True
        
        # On réduit le LR pour ne pas détruire les poids pré-entraînés
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
        status = "Fine-tuning"

    # --- ENTRAÎNEMENT ---
    # Note : assurez-vous que votre fonction train_one_epoch utilise bien l'optimizer actuel
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    
    scheduler.step() # Optionnel selon votre scheduler
    
    # --- VALIDATION ---
    val_loss, val_f1_macro, val_f1_per_class = validate()

    # --- LOGGING ---
    print(f"\nEpoch {epoch+1}/{EPOCHS} [{status}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | F1 Macro: {train_f1_macro:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val F1 Macro: {val_f1_macro:.4f}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro,
            'status': status
        })

    # --- SAUVEGARDE DU MEILLEUR MODÈLE ---
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" 🔥 Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

  0%|          | 0/129 [00:00<?, ?it/s]

100%|██████████| 129/129 [01:35<00:00,  1.35it/s]



Epoch 1/25 [Frozen]
Train Loss: 3.8750 | Train Acc: 0.0363 | F1 Macro: 0.0358
Val   Loss: 3.8193 | Val F1 Macro: 0.0327
 🔥 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.0327


100%|██████████| 129/129 [01:38<00:00,  1.31it/s]



Epoch 2/25 [Frozen]
Train Loss: 3.7522 | Train Acc: 0.1254 | F1 Macro: 0.1261
Val   Loss: 3.7062 | Val F1 Macro: 0.0829
 🔥 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.0829


...


Epoch 9/25 [Frozen]
Train Loss: 0.3250 | Train Acc: 0.9123 | F1 Macro: 0.9094
Val   Loss: 0.9049 | Val F1 Macro: 0.5901
 🔥 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.5901


100%|██████████| 129/129 [01:52<00:00,  1.14it/s]



Epoch 10/25 [Frozen]
Train Loss: 0.2697 | Train Acc: 0.9282 | F1 Macro: 0.9278
Val   Loss: 0.8770 | Val F1 Macro: 0.6145
 🔥 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.6145


 61%|██████    | 79/129 [01:12<00:45,  1.10it/s]


KeyboardInterrupt: 

In [ ]:
import csv
best_f1 = 0.0
best_model_path = "best_model.pth"

# Configuration du logging CSV
csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro', 'val_loss', 'val_f1_macro']

# Initialisation du fichier (écrase le précédent s'il existe)
with open(csv_file, mode='w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

for epoch in range(EPOCHS):
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()

    scheduler.step()
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")
    print(f"Train F1 per class: {train_f1_per_class}")

    val_loss, val_f1_macro, val_f1_per_class = validate()
    print(f"Val   Loss: {val_loss:.4f}")
    print(f"Val   F1 Macro: {val_f1_macro:.4f}")
    print(f"Val   F1 per class: {val_f1_per_class}")

    with open(csv_file, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({
            'epoch': epoch + 1,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'train_f1_macro': train_f1_macro,
            'val_loss': val_loss,
            'val_f1_macro': val_f1_macro
        })

    # 🔹 Sauvegarde du meilleur modèle
    if val_f1_macro > best_f1:
        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)
        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")

100%|██████████| 129/129 [01:45<00:00,  1.23it/s]


Epoch 1/25
Train Loss: 3.7560 | Train Acc: 0.1339
Train F1 Macro: 0.1249
Train F1 per class: [np.float64(0.208955223880597), np.float64(0.407185628742515), np.float64(0.05128205128205128), np.float64(0.09266409266409267), np.float64(0.07434944237918216), 0, np.float64(0.029268292682926828), np.float64(0.2105263157894737), np.float64(0.3818181818181818), np.float64(0.12363636363636364), np.float64(0.3876651982378855), np.float64(0.0617283950617284), np.float64(0.06470588235294118), np.float64(0.20074349442379183), np.float64(0.07058823529411765), np.float64(0.07883817427385892), np.float64(0.024242424242424242), np.float64(0.12472160356347439), np.float64(0.047430830039525695), np.float64(0.11956521739130434), np.float64(0.26332288401253917), np.float64(0.013071895424836603), np.float64(0.08771929824561404), np.float64(0.054644808743169404), np.float64(0.02666666666666667), np.float64(0.08333333333333333), np.float64(0.11055276381909547), np.float64(0.08187134502923976), np.float64(0.0

Val   Loss: 3.6764
Val   F1 Macro: 0.1033
Val   F1 per class: [0, np.float64(0.33333333333333337), np.float64(0.061538461538461535), 0, np.float64(0.18181818181818182), np.float64(0.022988505747126436), 0, 0, np.float64(0.6666666666666666), np.float64(0.1818181818181818), 0, 0, np.float64(0.1134020618556701), 0, np.float64(0.07142857142857142), np.float64(0.03333333333333333), 0, np.float64(0.12682926829268293), np.float64(0.1797752808988764), np.float64(0.47058823529411764), np.float64(0.13333333333333333), 0, np.float64(0.29245283018867924), np.float64(0.1142857142857143), 0, np.float64(0.24603174603174607), 0, 0, np.float64(0.022727272727272724), 0, 0, 0, np.float64(0.14925373134328357), 0, np.float64(0.07692307692307693), 0, np.float64(0.2170542635658915), np.float64(0.19354838709677416), 0, np.float64(0.03125), np.float64(0.28440366972477066), np.float64(0.06896551724137931), 0, np.float64(0.09411764705882353), np.float64(0.28571428571428575), np.float64(0.125), 0, 0, 0, np.float6

 56%|█████▌    | 72/129 [01:00<00:47,  1.20it/s]


KeyboardInterrupt: 

## **6. Création du fichier submission**

In [60]:
from torch.utils.data import DataLoader
import pandas as pd
import torch
from lib.data.dataset import BeeDataset

def submission(model, batch_size=32, transform=None, model_path="best_model.pth", save_path="submission.csv"):
    # Charger le modèle sur le bon device
    model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))
    model.to(DEVICE)
    model.eval()
    
    # Dataset de test
    test_dataset = BeeDataset(train=False, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    all_ids = []
    all_preds = []
    
    with torch.no_grad():
        for imgs, ids in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            
            preds = torch.argmax(outputs, dim=1)
            
            # Convertir preds en int et ids en int ou str
            all_preds.extend(preds.cpu().tolist())
            all_ids.extend([int(x) if isinstance(x, torch.Tensor) else x for x in ids])
    
    submission_df = pd.DataFrame({
        "id": all_ids,
        "label": all_preds
    })
    
    submission_df.to_csv(save_path, index=False)
    print(f"Submission saved to {save_path}")

In [62]:
test_dataset = BeeDataset(train=False, transform=val_preprocessor)

submission(model, batch_size=32, transform=val_preprocessor, save_path="submission2.csv")

/tmp/ipykernel_13960/2135808320.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=torch.device(DEVICE)))


Submission saved to submission2.csv
